# Lab Cycle 2 — Computational Linguistics Lab
**Course:** 23-813-0704 &nbsp;|&nbsp; **Date:** 28/07/2026

Six problems: minimum edit distance, a bigram-LM spell checker, a noisy-channel spell checker, a logistic-regression real-word (homophone) corrector, a small neural comparison model, and Naive Bayes sentiment analysis with add-k smoothing.

## Problem 7 — Minimum Edit Distance

Standard DP table with insertion cost 1, deletion cost 1, substitution cost 2 (the Jurafsky & Martin convention — a substitution is modeled as a delete+insert, so it costs double). Backtrace reconstructs the actual operations, not just the final number.

In [1]:
def min_edit_distance(source, target, ins_cost=1, del_cost=1, sub_cost=2):
    n, m = len(source), len(target)
    D = [[0] * (m + 1) for _ in range(n + 1)]
    for i in range(1, n + 1):
        D[i][0] = D[i - 1][0] + del_cost
    for j in range(1, m + 1):
        D[0][j] = D[0][j - 1] + ins_cost
    for i in range(1, n + 1):
        for j in range(1, m + 1):
            sub = D[i - 1][j - 1] + (0 if source[i - 1] == target[j - 1] else sub_cost)
            D[i][j] = min(D[i - 1][j] + del_cost, D[i][j - 1] + ins_cost, sub)
    return D


def backtrace(source, target, D, ins_cost=1, del_cost=1, sub_cost=2):
    i, j = len(source), len(target)
    ops = []
    while i > 0 or j > 0:
        if i > 0 and j > 0 and source[i - 1] == target[j - 1] and D[i][j] == D[i - 1][j - 1]:
            ops.append(("MATCH", source[i - 1], target[j - 1]))
            i -= 1; j -= 1
        elif i > 0 and j > 0 and D[i][j] == D[i - 1][j - 1] + sub_cost:
            ops.append(("SUB", source[i - 1], target[j - 1]))
            i -= 1; j -= 1
        elif i > 0 and D[i][j] == D[i - 1][j] + del_cost:
            ops.append(("DEL", source[i - 1], None))
            i -= 1
        elif j > 0 and D[i][j] == D[i][j - 1] + ins_cost:
            ops.append(("INS", None, target[j - 1]))
            j -= 1
        else:
            break
    ops.reverse()
    return ops


def show(source, target):
    D = min_edit_distance(source, target)
    ops = backtrace(source, target, D)
    print(f"{source} -> {target}: distance = {D[len(source)][len(target)]}")
    for op, a, b in ops:
        if op == "MATCH":
            print(f"  MATCH   {a}")
        elif op == "SUB":
            print(f"  SUB     {a} -> {b}")
        elif op == "DEL":
            print(f"  DEL     {a}")
        elif op == "INS":
            print(f"  INS     {b}")
    print()

show("intention", "execution")   # canonical J&M example -> distance 8
show("kitten", "sitting")
show("gray", "grey")

intention -> execution: distance = 8
  DEL     i
  SUB     n -> e
  SUB     t -> x
  MATCH   e
  INS     c
  SUB     n -> u
  MATCH   t
  MATCH   i
  MATCH   o
  MATCH   n

kitten -> sitting: distance = 5
  SUB     k -> s
  MATCH   i
  MATCH   t
  MATCH   t
  SUB     e -> i
  MATCH   n
  INS     g

gray -> grey: distance = 2
  MATCH   g
  MATCH   r
  SUB     a -> e
  MATCH   y



## Problem 8 — Bigram-LM spell checker

A small corpus about hats and hot weather, deliberately built so `hat` and `hot` are both valid English words at edit distance 1 from each other — perfect for showing that context, not just "is it a real word," is what a spell checker needs. Steps (a)-(e): tokenize + vocabulary, bigram counts, non-word detection, edit-distance-1 candidate generation, and ranking candidates by the probability they give the whole sentence under the bigram LM (add-1 smoothed).

In [2]:
import re
import math
import string
from collections import Counter

CORPUS_SENTENCES = [
    "she wore a hat to the party last night",
    "he wore a black hat every single day",
    "the hat was too big for her head",
    "put on your hat before you go outside",
    "she likes to wear a nice hat",
    "he found an old hat in the closet",
    "the magician pulled a rabbit from his hat",
    "my hat blew away in the wind",
    "he lifted his hat to greet everyone",
    "the weather today is very hot",
    "it was a hot day at the beach",
    "the soup tasted far too hot",
    "summer days can be extremely hot",
    "the sun made the sand very hot",
    "the coffee was still hot when i drank it",
    "the oven felt so hot to touch",
    "the pavement outside felt way too hot",
]

def tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

def sentence_tokens(s):
    return ["<s>"] + tokenize(s) + ["</s>"]

# (a) vocabulary
vocab = set()
unigram_counts = Counter()
# (b) bigram frequency table
bigram_counts = Counter()
for s in CORPUS_SENTENCES:
    toks = sentence_tokens(s)
    for t in toks:
        vocab.add(t); unigram_counts[t] += 1
    for w1, w2 in zip(toks, toks[1:]):
        bigram_counts[(w1, w2)] += 1

V = len(vocab)
print(f"Vocabulary size (incl. <s>/</s>): {V}")
print(f"Unique bigrams observed: {len(bigram_counts)}")

def bigram_prob(w1, w2):
    return (bigram_counts[(w1, w2)] + 1) / (unigram_counts[w1] + V)   # add-1 smoothing

def sentence_logprob(tokens):
    return sum(math.log(bigram_prob(w1, w2)) for w1, w2 in zip(tokens, tokens[1:]))

# (d) candidate generation: classic edits1 (insert/delete/substitute/transpose)
LETTERS = string.ascii_lowercase

def edits1(word):
    splits = [(word[:i], word[i:]) for i in range(len(word) + 1)]
    deletes = [a + b[1:] for a, b in splits if b]
    transposes = [a + b[1] + b[0] + b[2:] for a, b in splits if len(b) > 1]
    replaces = [a + c + b[1:] for a, b in splits if b for c in LETTERS]
    inserts = [a + c + b for a, b in splits for c in LETTERS]
    return set(deletes + transposes + replaces + inserts)

def candidates(word):
    return {w for w in edits1(word) if w in vocab}

# (c)+(e) detect non-word errors and rank candidates by sentence probability
def correct_sentence_bigram(sentence):
    toks = tokenize(sentence)
    errors = [t for t in toks if t not in vocab]
    print(f"Input: {sentence}")
    print(f"Non-word errors found: {errors}")
    for err in errors:
        cands = sorted(candidates(err))
        print(f"  Candidates for {err!r} (edit distance 1, in vocab): {cands}")
        scored = []
        for cand in cands:
            trial = ["<s>"] + [cand if t == err else t for t in toks] + ["</s>"]
            scored.append((cand, sentence_logprob(trial)))
        scored.sort(key=lambda x: -x[1])
        for cand, lp in scored:
            print(f"    {cand:8s} sentence log-prob = {lp:.3f}")
        if scored:
            print(f"  -> best correction: {err!r} => {scored[0][0]!r}")
    return errors

correct_sentence_bigram("she wore a hpt to the party")

Vocabulary size (incl. <s>/</s>): 79
Unique bigrams observed: 122
Input: she wore a hpt to the party
Non-word errors found: ['hpt']
  Candidates for 'hpt' (edit distance 1, in vocab): ['hat', 'hot']
    hat      sentence log-prob = -29.516
    hot      sentence log-prob = -29.910
  -> best correction: 'hpt' => 'hat'


## Problem 9 — Noisy channel spell checker

Same corpus and candidates as Problem 8, but ranked by `P(word) x P(typo|word)` instead of sentence context. `P(typo|word)` is a simplified channel model: a base rate per edit type (substitution most likely, then deletion, insertion, transposition rarest — illustrative constants, not corpus-derived), with substitutions further weighted by simplified QWERTY key adjacency, since a mistyped adjacent key is a far more plausible accident than a random one.

In [3]:
unigram_totals = sum(unigram_counts.values())

def unigram_prob(word, k=1):
    return (unigram_counts[word] + k) / (unigram_totals + k * V)

EDIT_TYPE_PRIOR = {"sub": 0.65, "del": 0.17, "ins": 0.13, "transpose": 0.05}

ADJACENT = {
    'q': 'wa', 'w': 'qeas', 'e': 'wrsd', 'r': 'etdf', 't': 'ryfg',
    'y': 'tugh', 'u': 'yihj', 'i': 'uojk', 'o': 'ipkl', 'p': 'ol',
    'a': 'qwsz', 's': 'awedxz', 'd': 'serfcx', 'f': 'drtgvc', 'g': 'ftyhbv',
    'h': 'gyujnb', 'j': 'huikmn', 'k': 'jiolm', 'l': 'kop',
    'z': 'asx', 'x': 'zsdc', 'c': 'xdfv', 'v': 'cfgb', 'b': 'vghn',
    'n': 'bhjm', 'm': 'njk',
}

def classify_edit(typo, candidate):
    if len(typo) == len(candidate):
        diffs = [(a, b) for a, b in zip(typo, candidate) if a != b]
        if len(diffs) == 1:
            typed, intended = diffs[0]
            return "sub", typed, intended
        for i in range(len(typo) - 1):
            if (typo[i], typo[i + 1]) == (candidate[i + 1], candidate[i]):
                return "transpose", None, None
    elif len(typo) == len(candidate) - 1:
        return "del", None, None
    elif len(typo) == len(candidate) + 1:
        return "ins", None, None
    return "unknown", None, None

def channel_prob(typo, candidate):
    edit_type, typed, intended = classify_edit(typo, candidate)
    base = EDIT_TYPE_PRIOR.get(edit_type, 0.01)
    if edit_type == "sub" and typed is not None:
        return base * (0.8 if typed in ADJACENT.get(intended, "") else 0.2)
    return base

def noisy_channel_rank(typo, cands):
    scored = []
    for cand in cands:
        p_word = unigram_prob(cand)
        p_typo = channel_prob(typo, cand)
        scored.append((cand, p_word, p_typo, p_word * p_typo))
    scored.sort(key=lambda x: -x[3])
    return scored

typo = "hpt"
cands = sorted(candidates(typo))   # reuse Problem 8's candidate generator
print(f"Typo: {typo!r}   Candidates: {cands}\n")
print(f"{'candidate':10s} {'P(word)':>10s} {'P(typo|word)':>14s} {'score':>12s}")
for cand, p_word, p_typo, score in noisy_channel_rank(typo, cands):
    edit_type, typed, intended = classify_edit(typo, cand)
    print(f"{cand:10s} {p_word:10.5f} {p_typo:14.5f} {score:12.6f}   ({edit_type}: '{typed}'->'{intended}')")

best = noisy_channel_rank(typo, cands)[0][0]
print(f"\nNoisy channel best correction: {typo!r} => {best!r}")
print("Bigram-LM best correction (Problem 8):    'hpt' => 'hat'")

Typo: 'hpt'   Candidates: ['hat', 'hot']

candidate     P(word)   P(typo|word)        score
hot           0.03766        0.52000     0.019582   (sub: 'p'->'o')
hat           0.04184        0.13000     0.005439   (sub: 'p'->'a')

Noisy channel best correction: 'hpt' => 'hot'
Bigram-LM best correction (Problem 8):    'hpt' => 'hat'


## Problem 10 — Logistic regression for real-word (homophone) errors

Confusion sets: `{write, right, rite}`, `{peace, piece}`, `{their, there, they're}`. One shared binary classifier is trained on `(context, candidate) -> is this candidate correct here?`, generalizing across all three sets via four feature types: left/right bigram log-probability, unigram log-prior, and a heuristic syntactic-compatibility score (e.g. a modal/`to` before a verb-slot word favors `write`; a determiner before a noun-slot word favors `rite`/`peace`/`piece`). At inference, every confusion-set token in a sentence gets scored against all of its set-mates and swapped for whichever scores highest.

In [4]:
import numpy as np
from sklearn.linear_model import LogisticRegression

CONFUSION_SETS = [
    {"write", "right", "rite"},
    {"peace", "piece"},
    {"their", "there", "they're"},
]

CANDIDATE_POS = {
    "write": "VERB", "right": "ADJ", "rite": "NOUN",
    "peace": "NOUN", "piece": "NOUN",
    "their": "DET", "there": "ADV", "they're": "PRON_VERB",
}

TRAIN_SENTENCES = [
    ("i will write a letter to my friend", "write"),
    ("she wants to write a novel someday", "write"),
    ("please write your name on the form", "write"),
    ("he will write the report tonight", "write"),
    ("they write letters every week", "write"),
    ("turn right at the next corner", "right"),
    ("you are absolutely right about that", "right"),
    ("she raised her right hand", "right"),
    ("that is the right answer", "right"),
    ("everyone has the right to vote", "right"),
    ("the wedding is an important rite of passage", "rite"),
    ("the ancient rite was performed at dawn", "rite"),
    ("it was a sacred rite for the tribe", "rite"),
    ("the funeral rite lasted three days", "rite"),
    ("the two countries signed a peace treaty", "peace"),
    ("she wanted peace and quiet", "peace"),
    ("the soldiers finally found peace", "peace"),
    ("world peace is a difficult goal", "peace"),
    ("he prayed for peace", "peace"),
    ("can i have a piece of cake", "piece"),
    ("she lost a piece of the puzzle", "piece"),
    ("he wrote a piece for the newspaper", "piece"),
    ("this is my favorite piece of music", "piece"),
    ("the piece of furniture was too heavy", "piece"),
    ("the students left their books at home", "their"),
    ("i saw their car in the parking lot", "their"),
    ("the team celebrated their victory", "their"),
    ("that is their house on the corner", "their"),
    ("put the keys over there", "there"),
    ("there is a cat on the roof", "there"),
    ("we will meet there at noon", "there"),
    ("there are many students in the class", "there"),
    ("they're going to the movies tonight", "they're"),
    ("they're planning a trip to kerala", "they're"),
    ("i heard they're moving to a new city", "they're"),
]

def hc_tokenize(text):
    return re.findall(r"[a-z']+", text.lower())

hc_unigram = Counter(); hc_bigram = Counter(); hc_total = 0; hc_vocab = set()
for sent, _ in TRAIN_SENTENCES:
    toks = ["<s>"] + hc_tokenize(sent) + ["</s>"]
    for t in toks:
        hc_unigram[t] += 1; hc_vocab.add(t); hc_total += 1
    for w1, w2 in zip(toks, toks[1:]):
        hc_bigram[(w1, w2)] += 1
HC_V = len(hc_vocab)

def hc_unigram_logprob(word, k=1):
    return math.log((hc_unigram[word] + k) / (hc_total + k * HC_V))

def hc_bigram_logprob(w1, w2, k=1):
    return math.log((hc_bigram[(w1, w2)] + k) / (hc_unigram[w1] + k * HC_V))

MODAL_TO = {"will", "to", "can", "must", "shall", "would", "should", "may", "might"}
DETERMINERS = {"a", "the", "this", "that", "these", "those", "my", "your", "his",
                "her", "its", "our", "an"}
ING_VERBS = {"going", "planning", "moving", "working", "looking", "coming", "heading"}

def pos_compat(candidate, prev_word, next_word):
    pos = CANDIDATE_POS[candidate]
    score = 0
    if prev_word in MODAL_TO and pos == "VERB":
        score += 1
    if prev_word in DETERMINERS and pos in ("NOUN", "ADJ"):
        score += 1
    if next_word == "of" and pos == "NOUN":
        score += 1
    if pos == "PRON_VERB" and next_word in ING_VERBS:
        score += 1
    return score

def hc_features(candidate, prev_word, next_word):
    return [
        hc_bigram_logprob(prev_word, candidate),
        hc_bigram_logprob(candidate, next_word),
        hc_unigram_logprob(candidate),
        pos_compat(candidate, prev_word, next_word),
    ]

def confusion_set_for(word):
    for s in CONFUSION_SETS:
        if word in s:
            return s
    return None

X, y = [], []
for sent, correct in TRAIN_SENTENCES:
    toks = hc_tokenize(sent)
    idx = toks.index(correct)
    prev_word = toks[idx - 1] if idx > 0 else "<s>"
    next_word = toks[idx + 1] if idx + 1 < len(toks) else "</s>"
    for cand in confusion_set_for(correct):
        X.append(hc_features(cand, prev_word, next_word))
        y.append(1 if cand == correct else 0)

X = np.array(X); y = np.array(y)
print(f"Training examples: {len(y)}  (positives={sum(y)}, negatives={len(y)-sum(y)})")

hc_clf = LogisticRegression(max_iter=1000)
hc_clf.fit(X, y)
print(f"Training accuracy: {hc_clf.score(X, y):.3f}")

def correct_homophones(sentence):
    raw_toks = re.findall(r"[A-Za-z']+", sentence)
    lower_toks = [t.lower() for t in raw_toks]
    out = list(raw_toks)
    report = []
    for i, tok in enumerate(lower_toks):
        cset = confusion_set_for(tok)
        if cset is None:
            continue
        prev_word = lower_toks[i - 1] if i > 0 else "<s>"
        next_word = lower_toks[i + 1] if i + 1 < len(lower_toks) else "</s>"
        scored = []
        for cand in cset:
            p = hc_clf.predict_proba([hc_features(cand, prev_word, next_word)])[0][1]
            scored.append((cand, p))
        scored.sort(key=lambda x: -x[1])
        best, best_p = scored[0]
        report.append((tok, scored))
        if best != tok:
            out[i] = best if raw_toks[i][0].islower() else best.capitalize()
    return " ".join(out), report

TEST_SENTENCES = [
    "Please right your name on the form",
    "There books are on the table",
    "The two countries signed a piece treaty",
    "Turn write at the next corner",
    "He wants to peace together the puzzle",
]

for s in TEST_SENTENCES:
    corrected, report = correct_homophones(s)
    print(f"\nInput:     {s}")
    print(f"Corrected: {corrected}")
    for tok, scored in report:
        scores_str = ", ".join(f"{c}={p:.3f}" for c, p in scored)
        print(f"  '{tok}' candidates -> {scores_str}")

Training examples: 95  (positives=35, negatives=60)
Training accuracy: 1.000

Input:     Please right your name on the form
Corrected: Please write your name on the form
  'right' candidates -> write=0.790, rite=0.064, right=0.063

Input:     There books are on the table
Corrected: Their books are on the table
  'there' candidates -> their=0.367, they're=0.285, there=0.282

Input:     The two countries signed a piece treaty
Corrected: The two countries signed a peace treaty
  'piece' candidates -> peace=0.763, piece=0.490

Input:     Turn write at the next corner
Corrected: Turn right at the next corner
  'write' candidates -> right=0.790, rite=0.064, write=0.063

Input:     He wants to peace together the puzzle
Corrected: He wants to piece together the puzzle
  'peace' candidates -> piece=0.059, peace=0.059


## Problem 11 — Neural spelling correction, and how it compares

A real pretrained neural spell-checker (a fine-tuned BERT/T5 checkpoint, or a dedicated tool like NeuSpell) needs to download weights from a model hub such as huggingface.co — a host this sandboxed environment's network allowlist doesn't include, so nothing pretrained is actually reachable here. The honest substitute: a small multi-layer perceptron, a genuine (if tiny) neural network, trained from scratch on synthetic typos generated from the Problem 8/9 corpus, scoring the same `hat` vs `hot` candidates so all three methods can be compared directly on one example.

In [5]:
import random
from sklearn.neural_network import MLPClassifier

random.seed(3)

def char_overlap(a, b):
    n = min(len(a), len(b))
    matches = sum(1 for i in range(n) if a[i] == b[i])
    return matches / max(len(a), len(b))

def nn_features(typo, candidate, prev_word, next_word):
    return [
        math.log(bigram_prob(prev_word, candidate)),
        math.log(bigram_prob(candidate, next_word)),
        math.log(unigram_prob(candidate)),
        char_overlap(typo, candidate),
        abs(len(typo) - len(candidate)),
    ]

def random_typo(word):
    i = random.randrange(len(word))
    kind = random.choice(["sub", "del", "ins"])
    if kind == "sub":
        return word[:i] + random.choice(LETTERS) + word[i + 1:]
    elif kind == "del":
        return word[:i] + word[i + 1:]
    else:
        return word[:i] + random.choice(LETTERS) + word[i:]

# ---- synthetic training data: several random typos per hat/hot occurrence ----
def build_training_set(typos_per_occurrence):
    Xn, yn = [], []
    for s in CORPUS_SENTENCES:
        toks = tokenize(s)
        for i, tok in enumerate(toks):
            if tok not in CONTENT_WORDS:
                continue
            prev_word = toks[i - 1] if i > 0 else "<s>"
            next_word = toks[i + 1] if i + 1 < len(toks) else "</s>"
            for _ in range(typos_per_occurrence):
                typo = random_typo(tok)
                neighbors = candidates(typo) | {tok}
                for cand in neighbors:
                    Xn.append(nn_features(typo, cand, prev_word, next_word))
                    yn.append(1 if cand == tok else 0)
    return np.array(Xn), np.array(yn)

CONTENT_WORDS = {"hat", "hot"}

# how much does training-set size matter for this tiny network?
for n_typos in (1, 6):
    random.seed(3)
    Xn, yn = build_training_set(n_typos)
    mlp_tmp = MLPClassifier(hidden_layer_sizes=(8,), max_iter=3000, random_state=0)
    mlp_tmp.fit(Xn, yn)
    print(f"typos/occurrence={n_typos}: {len(yn)} examples, "
          f"training accuracy = {mlp_tmp.score(Xn, yn):.3f}")
    if n_typos == 6:
        mlp = mlp_tmp   # keep the well-trained version for the actual demo below
print()

typo = "hpt"
prev_word, next_word = "a", "to"   # same local context as "she wore a hpt to the party"
scored = []
for cand in sorted(candidates(typo)):
    p = mlp.predict_proba([nn_features(typo, cand, prev_word, next_word)])[0][1]
    scored.append((cand, p))
scored.sort(key=lambda x: -x[1])

print(f"\nTypo: {typo!r}")
for cand, p in scored:
    print(f"  {cand:6s} P(correct) = {p:.3f}")
print(f"Neural best correction: {typo!r} => {scored[0][0]!r}")

print("\nSide by side on the same typo:")
print("  Bigram LM      (Problem 8): 'hpt' => 'hat'   (uses local context, ignores which edit happened)")
print("  Noisy channel  (Problem 9): 'hpt' => 'hot'   (uses edit-type/keyboard evidence + word frequency, ignores context)")
print(f"  Small neural   (Problem 11): 'hpt' => '{scored[0][0]}'   (learns to combine context + surface-form features)")

typos/occurrence=1: 33 examples, training accuracy = 0.970
typos/occurrence=6: 253 examples, training accuracy = 0.996


Typo: 'hpt'
  hat    P(correct) = 0.777
  hot    P(correct) = 0.724
Neural best correction: 'hpt' => 'hat'

Side by side on the same typo:
  Bigram LM      (Problem 8): 'hpt' => 'hat'   (uses local context, ignores which edit happened)
  Noisy channel  (Problem 9): 'hpt' => 'hot'   (uses edit-type/keyboard evidence + word frequency, ignores context)
  Small neural   (Problem 11): 'hpt' => 'hat'   (learns to combine context + surface-form features)


**Comparing all three correctors on the same typo ("hpt" -> ? in "she wore a hpt to the party")**

The bigram LM and the small neural net agree (`hat`), while the noisy channel model disagrees (`hot`) — and the reason is visible directly in each method's inputs. The bigram LM and the MLP both include the local context (`a ___`, `___ to`) among their signals, and that context strongly favors `hat` (reinforced specifically by "he lifted his hat **to** greet everyone" in the corpus). The noisy channel model never looks at context at all — it only asks "which candidate is a frequent word, and which edit is a plausible typo," and since `p` sits next to `o` on a QWERTY keyboard but nowhere near `a`, its channel term overwhelms the small difference in raw word frequency.

Neither answer is "wrong" given what each model is allowed to see. This is the textbook tradeoff between the two families: context-based models (bigram, and by extension the neural net once it's given context features) win when the sentence around the error is informative; channel-based models win when the *shape* of the mistake itself is the more reliable signal — for instance a genuine keyboard slip with an ambiguous or absent context. A production spell checker typically combines both (context AND a real, corpus-derived confusion matrix) rather than picking one.

The neural model is also the one place in this notebook where training-set *size* visibly mattered, even if the effect here is modest: with 1 synthetic typo per occurrence (33 examples) training accuracy was 0.970; with 6 typos per occurrence (253 examples) it rose to 0.996. This is a small, contained problem, so even sparse data gets a network most of the way there — but the direction of the effect is exactly what the statistical LM and noisy-channel methods above don't have to worry about: they pulled a usable signal straight out of the raw 17-sentence corpus with no synthetic augmentation needed at all, while the neural model needed that augmentation step to close the gap.

## Problem 12 — Naive Bayes sentiment classifier with add-k smoothing

A small, clearly polarized movie-review-style corpus (10 positive / 10 negative training sentences, 6 held-out test sentences). Multinomial Naive Bayes with add-k smoothing on `P(word | class)`, compared at k = 0.25, 0.75, and 1.

In [6]:
from collections import defaultdict

NB_TRAIN = [
    ("i absolutely loved this movie", "pos"),
    ("what a wonderful and touching film", "pos"),
    ("the acting was brilliant and the story was great", "pos"),
    ("a truly amazing and inspiring experience", "pos"),
    ("i enjoyed every minute of it", "pos"),
    ("the director did a fantastic job", "pos"),
    ("great performances and a beautiful soundtrack", "pos"),
    ("this is one of the best films i have seen", "pos"),
    ("the plot was engaging and the ending was satisfying", "pos"),
    ("a delightful and charming story", "pos"),
    ("i hated this movie", "neg"),
    ("what a boring and dull film", "neg"),
    ("the acting was terrible and the story was awful", "neg"),
    ("a truly disappointing and tedious experience", "neg"),
    ("i regretted watching it", "neg"),
    ("the director did a poor job", "neg"),
    ("weak performances and an annoying soundtrack", "neg"),
    ("this is one of the worst films i have seen", "neg"),
    ("the plot was confusing and the ending was unsatisfying", "neg"),
    ("a dreadful and tiresome story", "neg"),
]

NB_TEST = [
    ("i loved the brilliant acting", "pos"),
    ("a wonderful and inspiring film", "pos"),
    ("great story and a fantastic ending", "pos"),
    ("i hated the boring plot", "neg"),
    ("a dull and disappointing experience", "neg"),
    ("terrible acting and an awful ending", "neg"),
]

def nb_tokenize(text):
    return re.findall(r"[a-z]+", text.lower())

class NaiveBayesSentiment:
    def __init__(self, k=1.0):
        self.k = k
        self.class_word_counts = defaultdict(Counter)
        self.class_totals = Counter()
        self.class_doc_counts = Counter()
        self.vocab = set()

    def fit(self, data):
        self.classes = sorted(set(label for _, label in data))
        for text, label in data:
            self.class_doc_counts[label] += 1
            for w in nb_tokenize(text):
                self.class_word_counts[label][w] += 1
                self.class_totals[label] += 1
                self.vocab.add(w)
        self.V = len(self.vocab)
        self.n_docs = len(data)

    def logprob(self, text, label):
        lp = math.log(self.class_doc_counts[label] / self.n_docs)
        for w in nb_tokenize(text):
            count = self.class_word_counts[label][w]
            denom = self.class_totals[label] + self.k * self.V
            lp += math.log((count + self.k) / denom)
        return lp

    def predict(self, text):
        scores = {c: self.logprob(text, c) for c in self.classes}
        return max(scores, key=scores.get), scores

def evaluate(k):
    model = NaiveBayesSentiment(k=k)
    model.fit(NB_TRAIN)
    rows, correct = [], 0
    for text, gold in NB_TEST:
        pred, scores = model.predict(text)
        ok = pred == gold
        correct += ok
        rows.append((text, gold, pred, ok, scores))
    return model, correct / len(NB_TEST), rows

print(f"{'k':>6s} {'accuracy':>10s}")
results = {}
for k in (0.25, 0.75, 1.0):
    model, acc, rows = evaluate(k)
    results[k] = (model, acc, rows)
    print(f"{k:6.2f} {acc:10.3f}")

for k in (0.25, 0.75, 1.0):
    model, acc, rows = results[k]
    print(f"\n--- k = {k} (accuracy {acc:.3f}) ---")
    for text, gold, pred, ok, scores in rows:
        mark = "OK " if ok else "ERR"
        margin = scores["pos"] - scores["neg"]
        print(f"  [{mark}] gold={gold:3s} pred={pred:3s} margin={margin:+.3f}  \"{text}\"")

print("\n--- Borderline case: does k change the actual prediction? ---")
borderline = "amazing effects in a disappointing story"
for k in (0.25, 0.75, 1.0):
    model = NaiveBayesSentiment(k=k)
    model.fit(NB_TRAIN)
    pred, scores = model.predict(borderline)
    margin = scores["pos"] - scores["neg"]
    print(f"  k={k:<5} pred={pred:3s} margin={margin:+.4f}  \"{borderline}\"")

     k   accuracy
  0.25      1.000
  0.75      1.000
  1.00      1.000

--- k = 0.25 (accuracy 1.000) ---
  [OK ] gold=pos pred=pos margin=+3.036  "i loved the brilliant acting"
  [OK ] gold=pos pred=pos margin=+3.248  "a wonderful and inspiring film"
  [OK ] gold=pos pred=pos margin=+3.799  "great story and a fantastic ending"
  [OK ] gold=neg pred=neg margin=-3.401  "i hated the boring plot"
  [OK ] gold=neg pred=neg margin=-3.190  "a dull and disappointing experience"
  [OK ] gold=neg pred=neg margin=-5.047  "terrible acting and an awful ending"

--- k = 0.75 (accuracy 1.000) ---
  [OK ] gold=pos pred=pos margin=+1.563  "i loved the brilliant acting"
  [OK ] gold=pos pred=pos margin=+1.754  "a wonderful and inspiring film"
  [OK ] gold=pos pred=pos margin=+2.179  "great story and a fantastic ending"
  [OK ] gold=neg pred=neg margin=-1.826  "i hated the boring plot"
  [OK ] gold=neg pred=neg margin=-1.635  "a dull and disappointing experience"
  [OK ] gold=neg pred=neg margin=-2.700

**Reading the k comparison**

All three k values reach 100% accuracy on the 6-sentence test set — not surprising, since the test sentences reuse strongly polarized vocabulary (`loved`, `brilliant`, `fantastic` vs. `hated`, `boring`, `terrible`) that appears clearly in one class during training. Accuracy alone doesn't distinguish the three smoothing values here; on a small, clean toy corpus like this it usually won't.

The *margins* do, though, and move exactly the way add-k smoothing predicts: for every test sentence, the gap between the positive and negative log-probabilities shrinks as k grows. "Terrible acting and an awful ending," for example, goes from a confident -5.047 at k=0.25 to a much more conservative -2.218 at k=1.0. Larger k redistributes more probability mass toward words that were never seen in a given class, which pulls every prediction closer to 50/50 — the classifications don't change here because the true signal is strong enough to survive it, but the *confidence* reported by the model absolutely does change.

The one case where k changes the actual decision, not just the confidence, is "amazing effects in a disappointing story" — a genuinely mixed-signal sentence with one strong word from each class. At k=0.25 it narrowly predicts negative (margin -0.008); at k=0.75 and k=1.0 it flips to positive (+0.033, +0.043). With so little smoothing, `disappointing`'s slightly higher raw frequency in the negative-training data dominates; with more smoothing, that raw-count advantage gets diluted enough that other, smaller cues in the sentence tip the balance back the other way. This is the practical argument for tuning k on a validation set rather than defaulting to 1: it rarely matters on clear-cut inputs, but it's exactly the borderline cases — the ones a classifier most needs to get right — where the choice actually bites.

---
All six problems from Lab Cycle 2 done, every cell executed for real. Problems 8, 9, and 11 deliberately share one running example (`hpt` -> `hat`/`hot`) so the three correction strategies can be compared directly rather than in the abstract. Cycle 3 (Viterbi, TF-IDF/PPMI, WSD) is due 11/8 whenever you're ready for it.